# Chapter 47 — 3D Motion and Its 2D Projection

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/2d_motion_from_3d.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This chapter answers: when a 3D point moves (or the camera moves), how does its image — a 2D projection on the camera plane — move? The whole chapter is one long derivation of the **motion field equation** $(47.8)$. Once you have it, every interesting motion phenomenon — vanishing points, parallax, focus of expansion, time-to-contact, the way zoom looks different from dolly — falls out as a special case.

We reproduce the chapter's ten figures (47.1–47.10) by encoding the underlying math in PyTorch and plotting motion fields with `matplotlib`. There are no real photographs in this chapter; every figure is a diagram or a sketch of a vector field.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3d projection

torch.set_default_dtype(torch.float32)
torch.manual_seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 47.1 — Perspective projection of a moving point

A 3D point $\mathbf{P} = (X, Y, Z)$ projects to image coordinates $\mathbf{p} = (x, y)$ through the pinhole camera with focal length $f$ (equation 47.9):

$$x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z}.$$

When $\mathbf{P}$ moves with velocity $\dot{\mathbf{P}} = (\dot X, \dot Y, \dot Z)$, the image point moves too. Differentiating the projection (quotient rule) gives the fundamental motion equation (47.1):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{1}{Z}\begin{bmatrix} f & 0 & -x \\ 0 & f & -y \end{bmatrix}\begin{bmatrix}\dot X\\\dot Y\\\dot Z\end{bmatrix}.$$

The $1/Z$ factor is the entire reason vision can recover depth from motion — closer points produce larger image-plane velocities than farther points moving the same way.

In [ ]:
def project(P, f=1.0):
    """Pinhole projection (Eq. 47.9). P shape: (..., 3) → (..., 2)."""
    X, Y, Z = P[..., 0], P[..., 1], P[..., 2]
    return torch.stack([f * X / Z, f * Y / Z], dim=-1)


def project_velocity(P, P_dot, f=1.0):
    """Image-plane velocity from 3D point + 3D velocity (Eq. 47.1)."""
    X, Y, Z = P[..., 0], P[..., 1], P[..., 2]
    Xd, Yd, Zd = P_dot[..., 0], P_dot[..., 1], P_dot[..., 2]
    x = f * X / Z
    y = f * Y / Z
    xd = (f * Xd - x * Zd) / Z
    yd = (f * Yd - y * Zd) / Z
    return torch.stack([xd, yd], dim=-1)

In [ ]:
# Figure 47.1 — a 3D point P projects to a 2D point p on the camera plane.
fig = plt.figure(figsize=(8, 4.2))
ax = fig.add_subplot(1, 2, 1, projection="3d")

P = np.array([1.4, 0.8, 3.0])
f_plot = 1.0
p_img = np.array([f_plot * P[0] / P[2], f_plot * P[1] / P[2], f_plot])

# Optical axis + image plane (a small square at Z = f).
ax.plot([0, 0], [0, 0], [0, P[2] + 0.5], "k--", lw=0.7)
sq = np.array([[-0.7, -0.5, f_plot], [0.7, -0.5, f_plot], [0.7, 0.5, f_plot], [-0.7, 0.5, f_plot], [-0.7, -0.5, f_plot]])
ax.plot(sq[:, 0], sq[:, 1], sq[:, 2], color="steelblue", lw=1)

ax.scatter(*P, color="crimson", s=40)
ax.text(P[0] + 0.1, P[1], P[2], r"$\mathbf{P}$", color="crimson")
ax.scatter(*p_img, color="crimson", s=30)
ax.text(p_img[0] + 0.05, p_img[1], p_img[2] + 0.05, r"$\mathbf{p}$", color="crimson")
ax.plot([0, P[0]], [0, P[1]], [0, P[2]], color="crimson", lw=0.8)
ax.scatter(0, 0, 0, color="k", s=20)
ax.text(0.05, 0.05, -0.2, "optical center", fontsize=8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D scene")
ax.view_init(elev=18, azim=-58)

ax2 = fig.add_subplot(1, 2, 2)
ax2.set_xlim(-0.7, 0.7); ax2.set_ylim(-0.5, 0.5)
ax2.set_aspect("equal")
ax2.axhline(0, color="k", lw=0.4); ax2.axvline(0, color="k", lw=0.4)
ax2.scatter(p_img[0], p_img[1], color="crimson", s=40)
ax2.annotate(r"$\mathbf{p}=(fX/Z,\,fY/Z)$", (p_img[0], p_img[1]), xytext=(8, 8), textcoords="offset points", color="crimson")
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_title("image plane")

fig.suptitle("Figure 47.1 — A 3D point projects to a 2D point on the camera plane", y=1.02)
plt.tight_layout()
plt.show()

## 47.2 — Two easy cases of equation 47.1

Equation 47.1 has three terms. Killing one at a time isolates the two simplest motion regimes:

**Motion parallel to the image plane** ($\dot Z = 0$, equation 47.2):
$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{f}{Z}\begin{bmatrix}\dot X\\\dot Y\end{bmatrix}.$$
Magnitude inversely proportional to depth — pure parallax.

**Motion along the optical axis** ($\dot X = \dot Y = 0$, equation 47.3):
$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = -\frac{\dot Z}{Z}\begin{bmatrix}x\\y\end{bmatrix}.$$
Motion is radial, scaled by the ratio $\dot Z / Z$ (the **time-to-contact**).

Figure 47.2 sketches both.

In [ ]:
# Figure 47.2 — two geometry sketches: parallel-to-plane and parallel-to-axis motion.
fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))

for ax, mode in zip(axes, ["lateral", "forward"]):
    ax.set_aspect("equal")
    ax.axis("off")
    ax.plot([0, 4.5], [0, 0], "k", lw=0.8)      # optical axis
    ax.plot([1, 1], [-1.2, 1.2], "steelblue", lw=1)  # image plane at Z=f
    ax.text(1.05, 1.25, "image plane", color="steelblue", fontsize=9)
    ax.scatter(0, 0, color="k", s=20)
    ax.text(-0.15, -0.25, "O", fontsize=10)
    if mode == "lateral":
        P = np.array([1.0, 3.6])
        Pn = np.array([1.5, 3.6])
        ax.annotate("", xy=Pn, xytext=P, arrowprops=dict(arrowstyle="->", color="crimson"))
        ax.text((P[0] + Pn[0]) / 2, P[1] + 0.15, r"$\dot X$", color="crimson")
        ax.set_title("Motion parallel to camera plane ($\\dot Z = 0$)")
    else:
        P = np.array([1.2, 3.5])
        Pn = np.array([1.2, 2.6])
        ax.annotate("", xy=Pn, xytext=P, arrowprops=dict(arrowstyle="->", color="crimson"))
        ax.text(P[0] + 0.1, (P[1] + Pn[1]) / 2, r"$\dot Z$", color="crimson")
        ax.set_title("Motion along optical axis ($\\dot X = \\dot Y = 0$)")
    ax.scatter(*P, color="crimson", s=40)
    ax.text(P[0] + 0.1, P[1] + 0.15, r"$\mathbf{P}$", color="crimson")
    p_img = np.array([P[0] / P[1], 1.0])  # using f=1; swapped (x, Z) -> (Z=1 column)
    ax.scatter(p_img[0], p_img[1], color="crimson", s=25)
    ax.plot([0, P[0]], [0, P[1]], color="crimson", lw=0.6)

fig.suptitle("Figure 47.2 — Two motion regimes from equation 47.1")
plt.tight_layout()
plt.show()

## 47.2.1 — The vanishing point

Let a point move with constant world velocity $\mathbf{V} = (V_X, V_Y, V_Z)$ from initial position $\mathbf{P}_0$. Its position is $\mathbf{P}(t) = \mathbf{P}_0 + t\mathbf{V}$, and its image is

$$\mathbf{p}(t) = \frac{f}{P_{0,Z} + tV_Z}\begin{bmatrix}P_{0,X} + tV_X \\ P_{0,Y} + tV_Y\end{bmatrix}.$$

As $t \to \infty$ the initial-position terms drop out and we land at the **vanishing point** (equation 47.4):

$$x_{\infty} = f\,\frac{V_X}{V_Z},\qquad y_{\infty} = f\,\frac{V_Y}{V_Z}.$$

The vanishing point depends only on the *direction* of motion — never on where the point started. Gibson's bird flies away on a straight line; in the image its trajectory curves toward a single point on the horizon and never reaches it (figure 47.3).

In [ ]:
# Figure 47.3 — Gibson's bird: a 3D point flying along a straight line converges to a vanishing point.
f_plot = 1.0
P0 = torch.tensor([-1.5, 0.6, 2.0])
V = torch.tensor([0.5, 0.0, 1.0])
t = torch.linspace(0, 30, 80)
trajectory = P0[None] + t[:, None] * V[None]
p = project(trajectory, f=f_plot)
vp = torch.tensor([f_plot * V[0] / V[2], f_plot * V[1] / V[2]])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(p[:, 0], p[:, 1], "o-", color="crimson", markersize=3, lw=0.8, label="projected bird")
ax.scatter(vp[0], vp[1], color="navy", s=90, marker="x", label="vanishing point $(fV_X/V_Z,\\, fV_Y/V_Z)$")
ax.axhline(0, color="k", lw=0.3); ax.axvline(0, color="k", lw=0.3)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_xlim(-1.0, 1.0); ax.set_ylim(-0.5, 0.5)
ax.set_aspect("equal")
ax.set_title("Figure 47.3 — A bird flying away converges to the vanishing point")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 47.2.2 — The motion field under camera translation

When the *camera* translates with velocity $\mathbf{V}$ through a static world, each scene point moves with $-\mathbf{V}$ in the camera frame. Substituting $\dot{\mathbf{P}} = -\mathbf{V}$ into equation 47.1 gives the camera-translation motion field (equation 47.5):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \frac{1}{Z}\begin{bmatrix}-f & 0 & x\\0 & -f & y\end{bmatrix}\begin{bmatrix}V_X\\V_Y\\V_Z\end{bmatrix}.$$

The full motion field including rotation is given by equation 47.8, which we encode once below and reuse for every figure that follows.

In [ ]:
def motion_field(x, y, Z, V, W, f=1.0):
    """Full image-plane motion field (Eq. 47.8).

    Args:
        x, y: image-plane coordinates, same shape.
        Z:    scene depth at each (x, y), broadcastable to x.
        V:    (3,) camera translational velocity (V_X, V_Y, V_Z).
        W:    (3,) camera angular velocity (W_X, W_Y, W_Z).
        f:    focal length.
    Returns:
        (xdot, ydot): same shape as x.
    """
    Vx, Vy, Vz = V
    Wx, Wy, Wz = W
    trans_x = (-f * Vx + x * Vz) / Z
    trans_y = (-f * Vy + y * Vz) / Z
    rot_x = (x * y * Wx - (f * f + x * x) * Wy + f * y * Wz) / f
    rot_y = ((f * f + y * y) * Wx - x * y * Wy - f * x * Wz) / f
    return trans_x + rot_x, trans_y + rot_y


def image_grid(nx=15, ny=11, half_x=1.0, half_y=0.7):
    """A regular grid of image-plane sample points for quiver plots."""
    xs = torch.linspace(-half_x, half_x, nx)
    ys = torch.linspace(-half_y, half_y, ny)
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    return gx, gy

### 47.2.2.1 — Lateral camera motion

Driving past a roadside scene: $V_X \ne 0$, $V_Y = V_Z = 0$. The motion field collapses to

$$\dot x = -\frac{f V_X}{Z}, \qquad \dot y = 0.$$

Image-plane speed is inversely proportional to scene depth: close objects whip past while distant clouds barely move (figure 47.5).

In [ ]:
# Figure 47.4 — sketch: lateral camera motion parallel to the image plane.
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.axis("off"); ax.set_aspect("equal")
ax.plot([0, 4.5], [0, 0], "k", lw=0.7)
ax.plot([1, 1], [-1.0, 1.0], "steelblue", lw=1)
ax.text(1.05, 1.05, "image plane", color="steelblue", fontsize=9)
ax.scatter(0, 0, color="k", s=20)
ax.annotate("", xy=(0.6, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="crimson"))
ax.text(0.25, 0.12, r"$V_X$", color="crimson")
for Z in [2.0, 3.0, 4.0]:
    ax.scatter(Z, 0.8, color="gray", s=30)
    ax.text(Z + 0.05, 0.85, f"Z={Z:g}", fontsize=8, color="gray")
ax.set_title("Figure 47.4 — Lateral camera motion")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 47.5 — motion field under lateral camera translation.
# Depth varies vertically in the scene: foreground at bottom, sky/clouds at top.
gx, gy = image_grid(nx=15, ny=11)
Z = 2.0 + 6.0 * (gy + 0.7) / 1.4   # closer at bottom of image, farther at top
V = (1.0, 0.0, 0.0); W = (0.0, 0.0, 0.0)
u, v = motion_field(gx, gy, Z, V, W, f=1.0)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.quiver(gx.numpy(), gy.numpy(), u.numpy(), v.numpy(),
          angles="xy", scale_units="xy", scale=2.5, color="crimson", width=0.004)
ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.8, 0.8); ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Figure 47.5 — Lateral motion field (close objects move faster)")
plt.tight_layout()
plt.show()

### 47.2.2.2 — Forward camera motion and the focus of expansion

Driving toward a wall: $V_X = V_Y = 0$, $V_Z > 0$. Equation 47.5 simplifies to

$$\dot x = \frac{V_Z}{Z}\,x, \qquad \dot y = \frac{V_Z}{Z}\,y,$$

a radial flow centered at the origin. The point at which the field is zero is the **focus of expansion** — and the scaling factor $V_Z / Z$ is the inverse **time-to-contact**, the quantity a fly's-eye control loop reads off to time a landing.

In [ ]:
# Figure 47.6 — sketch: camera moving forward along its optical axis.
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.axis("off"); ax.set_aspect("equal")
ax.plot([0, 4.5], [0, 0], "k", lw=0.7)
ax.plot([1, 1], [-1.0, 1.0], "steelblue", lw=1)
ax.text(1.05, 1.05, "image plane", color="steelblue", fontsize=9)
ax.plot([3.8, 3.8], [-0.9, 0.9], "gray", lw=1)
ax.text(3.85, 0.95, "wall", color="gray", fontsize=9)
ax.scatter(0, 0, color="k", s=20)
ax.annotate("", xy=(0, 0), xytext=(-0.7, 0), arrowprops=dict(arrowstyle="<-", color="crimson"))
ax.text(-0.5, 0.12, r"$V_Z$", color="crimson")
ax.set_title("Figure 47.6 — Camera moving forward")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 47.7 — motion field as the camera approaches a planar surface (constant Z).
gx, gy = image_grid(nx=15, ny=11)
Z = torch.full_like(gx, 4.0)            # planar surface perpendicular to optical axis
V = (0.0, 0.0, 1.0); W = (0.0, 0.0, 0.0)
u, v = motion_field(gx, gy, Z, V, W, f=1.0)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.quiver(gx.numpy(), gy.numpy(), u.numpy(), v.numpy(),
          angles="xy", scale_units="xy", scale=2.5, color="crimson", width=0.004)
ax.scatter(0, 0, color="navy", s=90, marker="x")
ax.annotate("focus of expansion", (0, 0), xytext=(10, -16), textcoords="offset points", color="navy", fontsize=9)
ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.8, 0.8); ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Figure 47.7 — Forward motion: radial expansion from the focus of expansion")
plt.tight_layout()
plt.show()

## 47.2.3 — Camera rotation and the full motion field

Adding rotation, the point's velocity relative to a rotating camera with angular velocity $\mathbf{\Omega}$ is

$$\dot{\mathbf{P}} = -\mathbf{V} - \mathbf{\Omega} \times \mathbf{P}.$$

Plugging this into equation 47.1 produces the full motion field (equation 47.8):

$$\begin{bmatrix}\dot x\\\dot y\end{bmatrix} = \underbrace{\frac{1}{Z}\begin{bmatrix}-f & 0 & x\\0 & -f & y\end{bmatrix}\begin{bmatrix}V_X\\V_Y\\V_Z\end{bmatrix}}_{\text{translation: depends on }Z} \;+\; \underbrace{\frac{1}{f}\begin{bmatrix}xy & -(f^2+x^2) & fy\\ f^2+y^2 & -xy & -fx\end{bmatrix}\begin{bmatrix}\Omega_X\\\Omega_Y\\\Omega_Z\end{bmatrix}}_{\text{rotation: independent of }Z}.$$

Two qualitative consequences are worth noting:

* Rotational flow is **independent of depth** — it tells you nothing about the scene's 3D structure.
* Translational flow scales as $1/Z$ — it carries all the depth information.

Figure 47.8 introduces the three rotation axes (yaw, pitch, roll) used throughout the rest of the chapter.

In [ ]:
# Figure 47.8 — Euler angles: yaw around Z (optical), pitch around X, roll around Y. Convention follows the book.
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection="3d")
ax.quiver(0, 0, 0, 1, 0, 0, color="crimson", arrow_length_ratio=0.1)
ax.quiver(0, 0, 0, 0, 1, 0, color="seagreen", arrow_length_ratio=0.1)
ax.quiver(0, 0, 0, 0, 0, 1, color="steelblue", arrow_length_ratio=0.1)
ax.text(1.05, 0, 0, r"$X$ (pitch)", color="crimson")
ax.text(0, 1.05, 0, r"$Y$ (roll)", color="seagreen")
ax.text(0, 0, 1.05, r"$Z$ (yaw)", color="steelblue")
# curved arrows hinting at rotation around each axis
theta = np.linspace(0, 2 * np.pi, 60)
ax.plot(0.3 * np.cos(theta), 0.3 * np.sin(theta), zs=0, color="steelblue", lw=0.7)
ax.plot(np.zeros_like(theta), 0.3 * np.cos(theta), 0.3 * np.sin(theta), color="crimson", lw=0.7)
ax.plot(0.3 * np.cos(theta), np.zeros_like(theta), 0.3 * np.sin(theta), color="seagreen", lw=0.7)
ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
ax.set_box_aspect((1, 1, 1))
ax.set_title("Figure 47.8 — Euler angles (yaw, pitch, roll)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Rotation around the optical axis ($\Omega_X = \Omega_Y = 0$)

With only $\Omega_Z$ active, equation 47.8 collapses to

$$\dot x = \Omega_Z\,y,\qquad \dot y = -\Omega_Z\,x,$$

the velocity field of rigid rotation about the image origin. Every concentric circle is an integral curve (figure 47.9).

In [ ]:
# Figure 47.9 — pure optical-axis rotation produces a circular motion field.
gx, gy = image_grid(nx=15, ny=11)
Z = torch.full_like(gx, 5.0)
V = (0.0, 0.0, 0.0); W = (0.0, 0.0, 1.0)
u, v = motion_field(gx, gy, Z, V, W, f=1.0)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.quiver(gx.numpy(), gy.numpy(), u.numpy(), v.numpy(),
          angles="xy", scale_units="xy", scale=2.0, color="crimson", width=0.004)
ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.8, 0.8); ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Figure 47.9 — Rotation around the optical axis ($\\Omega_Z$)")
plt.tight_layout()
plt.show()

## 47.2.4 — Rotation under varying focal length

Y-axis rotation with $\Omega_X = \Omega_Z = 0$ produces

$$\dot x = -\frac{(f^2 + x^2)}{f}\,\Omega_Y,\qquad \dot y = -\frac{xy}{f}\,\Omega_Y,$$

which depends strongly on $f$. Wide-angle lenses ($f$ small) produce flows with sharp curvature near the edges; long lenses ($f$ large) produce nearly uniform horizontal flow. Figure 47.10 sweeps $f \in \{1/3, 1, 3\}$ with the same scene and angular velocity.

In [ ]:
# Figure 47.10 — Y-axis rotation at three focal lengths.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, f_val in zip(axes, [1 / 3, 1.0, 3.0]):
    gx, gy = image_grid(nx=13, ny=11, half_x=1.0, half_y=0.7)
    Z = torch.full_like(gx, 5.0)
    V = (0.0, 0.0, 0.0); W = (0.0, 1.0, 0.0)
    u, v = motion_field(gx, gy, Z, V, W, f=f_val)
    ax.quiver(gx.numpy(), gy.numpy(), u.numpy(), v.numpy(),
              angles="xy", scale_units="xy", scale=8.0, color="crimson", width=0.005)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.8, 0.8); ax.set_aspect("equal")
    ax.set_title(f"$f = {f_val:.3g}$")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
fig.suptitle("Figure 47.10 — Y-axis rotation at three focal lengths")
plt.tight_layout()
plt.show()

## 47.3 — Concluding remarks

The chapter's whole arc is one of these three ideas:

1. **Differentiate the projection** $\mathbf{p} = (fX/Z, fY/Z)$ to get the image-plane motion of any moving 3D point (eq. 47.1).
2. **Substitute the camera's rigid-body kinematics** $\dot{\mathbf{P}} = -\mathbf{V} - \mathbf{\Omega}\times\mathbf{P}$ to get the motion field (eq. 47.8).
3. **Read off special cases**: vanishing points (47.4), depth-modulated parallax (47.5), focus of expansion and time-to-contact (47.6), depth-independent rotation (47.7), focal-length sensitivity (47.10).

The next chapter (48 — Optical Flow) flips the script: given two frames, *recover* the motion field. The geometry here is the forward model that chapter inverts.